In [1]:
import numpy as np
import pandas as pd
from scipy.signal import welch

# Load rebuilt dataset
windows = np.load("../data/windows.npy")
labels = np.load("../data/labels.npy")

print("Windows Shape:", windows.shape)
print("Labels Shape:", labels.shape)

Windows Shape: (13181, 23, 1024)
Labels Shape: (13181,)


In [2]:
# Frequency bands
frequency_bands = {
    "Delta": (0.5, 4),
    "Theta": (4, 8),
    "Alpha": (8, 13),
    "Beta": (13, 30),
    "Gamma": (30, 40)
}

sfreq = 256

print("Sampling Frequency:", sfreq, "Hz")
print("Frequency Bands:", frequency_bands)

Sampling Frequency: 256 Hz
Frequency Bands: {'Delta': (0.5, 4), 'Theta': (4, 8), 'Alpha': (8, 13), 'Beta': (13, 30), 'Gamma': (30, 40)}


In [3]:
def extract_statistical_features(window):
    
    # Calculate features for each EEG channel
    channel_means = np.mean(window, axis=1)
    channel_stds = np.std(window, axis=1)
    channel_variances = np.var(window, axis=1)

    # Average across all EEG channels
    mean_feature = np.mean(channel_means)
    std_feature = np.mean(channel_stds)
    variance_feature = np.mean(channel_variances)

    return mean_feature, std_feature, variance_feature

In [4]:
def extract_frequency_features(window, sfreq=256):

    channel_features = []

    for channel in window:

        # Calculate Power Spectral Density using Welch method
        frequencies, psd = welch(
            channel,
            fs=sfreq,
            nperseg=512
        )

        band_features = {}

        # Calculate power for each frequency band
        for band_name, (low_freq, high_freq) in frequency_bands.items():

            frequency_mask = (
                (frequencies >= low_freq) &
                (frequencies < high_freq)
            )

            band_power = np.trapezoid(
                psd[frequency_mask],
                frequencies[frequency_mask]
            )

            band_features[band_name] = band_power

        channel_features.append(band_features)

    # Convert channel features to DataFrame
    channel_features_df = pd.DataFrame(channel_features)

    # Average features across all 23 EEG channels
    window_frequency_features = channel_features_df.mean()

    return window_frequency_features

In [5]:
# Test feature extraction on the first EEG window

test_features = extract_statistical_features(windows[0])
frequency_features = extract_frequency_features(windows[0], sfreq)

print("Statistical Features:")
print("Mean:", test_features[0])
print("Std:", test_features[1])
print("Variance:", test_features[2])

print("\nFrequency Features:")
print(frequency_features)

Statistical Features:
Mean: 6.224838260383522e-06
Std: 3.986366016444262e-05
Variance: 2.0190720572477735e-09

Frequency Features:
Delta    5.086776e-10
Theta    1.252318e-10
Alpha    4.501589e-11
Beta     9.936528e-11
Gamma    5.169968e-11
dtype: float64


In [6]:
# Extract features from all EEG windows

all_features = []

for i, window in enumerate(windows):

    features_for_window = [
        *extract_statistical_features(window),
        *extract_frequency_features(window, sfreq).values
    ]

    all_features.append(features_for_window)

    if (i + 1) % 500 == 0:
        print(f"Processed {i + 1}/{len(windows)} windows")

# Convert to NumPy array
features = np.array(all_features)

print("\nFeature Extraction Completed!")
print("Feature Matrix Shape:", features.shape)

Processed 500/13181 windows
Processed 1000/13181 windows
Processed 1500/13181 windows
Processed 2000/13181 windows
Processed 2500/13181 windows
Processed 3000/13181 windows
Processed 3500/13181 windows
Processed 4000/13181 windows
Processed 4500/13181 windows
Processed 5000/13181 windows
Processed 5500/13181 windows
Processed 6000/13181 windows
Processed 6500/13181 windows
Processed 7000/13181 windows
Processed 7500/13181 windows
Processed 8000/13181 windows
Processed 8500/13181 windows
Processed 9000/13181 windows
Processed 9500/13181 windows
Processed 10000/13181 windows
Processed 10500/13181 windows
Processed 11000/13181 windows
Processed 11500/13181 windows
Processed 12000/13181 windows
Processed 12500/13181 windows
Processed 13000/13181 windows

Feature Extraction Completed!
Feature Matrix Shape: (13181, 8)


In [7]:
# Create feature names

feature_names = [
    "Mean",
    "Std",
    "Variance",
    "Delta",
    "Theta",
    "Alpha",
    "Beta",
    "Gamma"
]

# Create DataFrame
features_df = pd.DataFrame(
    features,
    columns=feature_names
)

# Add labels
features_df["Label"] = labels

# Display basic information
print("Final Dataset Shape:", features_df.shape)

print("\nFeature Columns:")
print(feature_names)

print("\nClass Distribution:")
print(features_df["Label"].value_counts().sort_index())

print("\nFirst 5 Rows:")
display(features_df.head())

Final Dataset Shape: (13181, 9)

Feature Columns:
['Mean', 'Std', 'Variance', 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']

Class Distribution:
Label
0    13080
1      101
Name: count, dtype: int64

First 5 Rows:


,Mean,Std,Variance,Delta,Theta,Alpha,Beta,Gamma,Label
0,6.224838e-06,0.000040,2.019072e-09,5.086776e-10,1.252318e-10,4.501589e-11,9.936528e-11,5.169968e-11,0
1,-6.238922e-07,0.000035,1.604471e-09,1.510739e-09,1.018136e-10,3.810161e-11,1.113378e-10,7.269863e-11,0
2,9.433753e-07,0.000047,3.111549e-09,2.225362e-09,2.902292e-10,5.649391e-11,1.265199e-10,9.276142e-11,0
3,2.200194e-07,0.000031,1.184050e-09,6.634073e-10,1.910103e-10,4.744904e-11,2.174806e-10,1.205579e-10,0
4,-1.851659e-07,0.000025,7.406582e-10,3.636297e-10,9.952621e-11,3.020521e-11,1.349834e-10,7.228049e-11,0


In [8]:
# Check for missing values

print("Missing Values in Each Column:")
print(features_df.isnull().sum())

print("\nTotal Missing Values:")
print(features_df.isnull().sum().sum())

Missing Values in Each Column:
Mean        0
Std         0
Variance    0
Delta       0
Theta       0
Alpha       0
Beta        0
Gamma       0
Label       0
dtype: int64

Total Missing Values:
0
